# DEAD proposal stuff

In [ ]:
import duckdb as db
import pandas as pd
import numpy as np
con = db.connect()

## Download + Parquet

In [ ]:
import duckdb

con = duckdb.connect()

# NOTE: this direct-from-URL approach failed with a CSV "sniffing" error.
# That error almost always means the URL is NOT returning a clean CSV --
# e.g. the Iowa data portal is returning an HTML error/redirect page,
# a JSON payload, or a gzip/paginated response instead of plain CSV.
# Before re-trying this, open one of the URLs in a browser (or curl -I it)
# and confirm you actually get a CSV file back, not a webpage.
#
# If you DO get a real CSV and it still fails to parse, try being explicit
# instead of relying on auto-detection, e.g.:
#
# con.execute("""
#     COPY (
#         SELECT *, 2022 AS source_year
#         FROM read_csv(
#             'https://idh-be.iowa.gov/api/v1/datasets/1259/rows.csv',
#             delim=',',
#             quote='\"',
#             ignore_errors=true,
#             max_line_size=10000000
#         )
#     )
#     TO 'liquor_2022.parquet'
#     (FORMAT PARQUET);
# """)
#
# For now we're using the manually-exported JSON approach below instead,
# which sidesteps this URL-parsing issue entirely.


What factors drive higher or lower alcohol sales?

In [ ]:
import zipfile
import glob
import os
import duckdb

# --- 1) Locate the zip file ---
# The FileNotFoundError means Python looked for the zip in your CURRENT
# WORKING DIRECTORY (wherever you launched Jupyter from) and didn't find it
# there. It does NOT mean the file doesn't exist anywhere on your machine --
# it's very likely still sitting in your Downloads folder (or wherever your
# OneDrive sync saves it), just not next to this notebook.

zip_name = "OneDrive_1_9-10-2026.zip"

# Places to check, in order. Add/edit paths here if yours lives somewhere else.
candidate_dirs = [
    ".",                                   # same folder as this notebook
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~/OneDrive"),
    os.path.expanduser("~/OneDrive/Downloads"),
]

zip_path = None
for d in candidate_dirs:
    candidate = os.path.join(d, zip_name)
    if os.path.isfile(candidate):
        zip_path = candidate
        break

if zip_path is None:
    # Last resort: search a couple levels down from the home directory
    matches = glob.glob(os.path.expanduser(f"~/**/{zip_name}"), recursive=True)
    if matches:
        zip_path = matches[0]

if zip_path is None:
    raise FileNotFoundError(
        f"Could not find '{zip_name}' automatically. "
        f"Current working directory is: {os.getcwd()}\n"
        f"Run `os.listdir('.')` to see what's actually here, find where the "
        f"zip downloaded to, and either move it next to this notebook or set "
        f"zip_path = r'/full/path/to/{zip_name}' manually below."
    )

print(f"Using zip file: {zip_path}")

extract_path = "liquor_2022_2026"

# --- 2) Unzip ---
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# --- 3) Find every JSON file inside all subfolders ---
json_files = glob.glob(f"{extract_path}/**/*.json", recursive=True)
print(f"Found {len(json_files)} JSON files")

if not json_files:
    raise FileNotFoundError(
        f"Unzipped '{zip_path}' into '{extract_path}' but found no .json files. "
        f"Check that this is the right export (JSON, not CSV/XLSX) and that "
        f"the zip actually contains data files and not just a single wrapper folder."
    )

# --- 4) Convert all JSON files into one Parquet ---
con = duckdb.connect()

con.execute(
    """
    COPY (
        SELECT *
        FROM read_json_auto(
            ?,
            union_by_name=true
        )
    )
    TO 'liquor_2022_2026.parquet'
    (FORMAT PARQUET)
    """,
    [json_files],
)

print("Done! Created liquor_2022_2026.parquet")

# --- 5) Quick sanity check: peek at the schema/columns you'll be working with ---
preview = con.execute("SELECT * FROM 'liquor_2022_2026.parquet' LIMIT 5").df()
print(preview.columns.tolist())
preview


## Inspect schema before aggregating

Column names below are my best guess based on the standard Iowa liquor sales schema (Socrata export). **Run this first and check the printed column list against the SQL below** — rename anything that doesn't match your actual file.

In [ ]:
import duckdb

con = duckdb.connect()

# Peek at the actual columns/types in your file
print(con.execute("DESCRIBE SELECT * FROM 'liquor_2022_2026.parquet'").df())
con.execute("SELECT * FROM 'liquor_2022_2026.parquet' LIMIT 3").df()


## Build the Category x Month aggregated table

Observational unit: **Alcohol Category x Year-Month**.

Note: I'm aggregating by *year-month* (e.g. `2023-06`), not just calendar month
(e.g. `June`) collapsed across years -- if you collapse across years you throw
away 2022 vs 2026 trend/growth, which is almost certainly something DEAD cares
about. If you actually want pure seasonality (one row per category per
calendar month, pooling all years together), swap `year_month` for `month_num`
in the GROUP BY -- I left `month_num` and `year` as separate columns either way
so you can slice it either direction later.

**Column names, confirmed against `DESCRIBE` output on the actual file:**
- `ordered_on` -> the sale date column (already a `DATE` type)
- `category_name` -> the liquor category text field
- `sales_bottles` -> unit count (stored as `VARCHAR`, cast to numeric)
- `sales_dollars` -> dollar amount (already `DOUBLE`)
- `sales_liters` -> volume (already `DOUBLE`)
- `state_bottle_retail` -> per-bottle retail price (stored as `VARCHAR`, cast to numeric)
- `store_no`, `store_zip_code` -> used only to build diversity/reach features, not as grouping keys


In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE agg AS
    SELECT
        category_name,
        strftime(ordered_on, '%Y-%m')                    AS year_month,
        EXTRACT(year  FROM ordered_on)                    AS year,
        EXTRACT(month FROM ordered_on)                    AS month_num,

        -- === targets ===
        SUM(TRY_CAST(sales_bottles AS DOUBLE))            AS total_units,
        SUM(sales_dollars)                                AS total_dollars,

        -- === supporting features, aggregated to the same grain ===
        SUM(sales_liters)                                 AS total_liters,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE))      AS avg_bottle_price,
        SUM(sales_dollars) / NULLIF(SUM(TRY_CAST(sales_bottles AS DOUBLE)), 0) AS realized_price_per_unit,
        COUNT(DISTINCT store_no)                          AS n_stores,
        COUNT(DISTINCT store_zip_code)                    AS n_zips_reached,
        COUNT(*)                                          AS n_transactions

    FROM 'liquor_2022_2026.parquet'
    GROUP BY category_name, year_month, year, month_num
    ORDER BY category_name, year_month
""")

df = con.execute("SELECT * FROM agg").df()
print(df.shape)
df.head(10)


## Target variable prep

Two candidate targets per row (`total_units`, `total_dollars`). As discussed:
`total_dollars` conflates volume with price/premiumization, so I'd treat
`total_units` (or `total_liters` if you want an even cleaner consumption
measure) as primary, and `total_dollars` as a secondary/parallel target if you
also want to speak to spending.

Both are log-transformed below (`log1p` handles any zero-sale category-months
safely) since raw sales counts are typically right-skewed across categories --
this will matter once you're checking linear regression residuals.


In [ ]:
import numpy as np

df["log_units"]   = np.log1p(df["total_units"])
df["log_dollars"] = np.log1p(df["total_dollars"])
df["log_liters"]  = np.log1p(df["total_liters"])

# Quick look at skew before/after log transform
print(df[["total_units", "log_units", "total_dollars", "log_dollars"]].describe())

df.to_parquet("liquor_category_month_agg.parquet", index=False)
print("Saved liquor_category_month_agg.parquet")


## Optional: seasonality / trend features for the regression

Since your obs unit is time-based (year-month per category), these are the
natural linear-regression-friendly features to add before modeling -- month
dummies to capture seasonality (holidays, summer, etc.) and a simple linear
time trend to capture year-over-year growth/decline, plus a category dummy
set since "category" itself is a huge driver of both units and price.

In [ ]:
# Linear time trend (months since the start of the panel)
df["t"] = (df["year"] - df["year"].min()) * 12 + df["month_num"]

# One-hot encode month (seasonality) and category (product mix) for use
# directly in a linear regression / OLS design matrix
model_df = pd.get_dummies(
    df,
    columns=["month_num", "category_name"],
    drop_first=True,
)

model_df.head()

## Weather (statewide, monthly) — testing the DEAD hypothesis

DEAD's question is what drives higher/lower alcohol purchases, and the
hypothesis on the table is **weather**. The observational unit built above
is *category x year-month*, statewide — there's no store/zip dimension left
at this grain — so weather needs to come in as a single **statewide monthly
weather series**, not the per-zip join used in `final_eda.ipynb` (that join
is the right one for transaction-level work; this notebook's grain is
coarser on purpose, for the regression setup below).

**Design decision, up front, because it changes what the regression can
actually tell DEAD:** `model_df` above already has a month-of-year dummy
for every calendar month, to soak up seasonality. Average temperature *is*
essentially a deterministic function of calendar month — January is cold
and July is warm every year — so if we hand the regression both "month =
July" and "average temp = 83°F" as separate predictors, they're almost
perfectly collinear and the model can't tell you which one is doing the
work. That's not really the question DEAD is asking anyway: "it's warmer
in summer" isn't an actionable weather finding, it's just seasonality
restated.

The more useful (and more rigorous) test is whether a given month's weather
being **unusual for that month** predicts sales being unusual for that
month — i.e. was this particular July hotter or wetter than a typical July,
and did that July sell more or less alcohol than a typical July. The IEM
daily feed conveniently ships a `climo_high_f` / `climo_low_f` /
`climo_precip_in` column alongside every observation — the NOAA long-run
climate normal for that station and calendar date — so the anomaly
(actual minus normal) is available directly rather than needing to be
estimated from this dataset's own 5 years of history.

Both cuts are fit below: a simpler model using raw weather levels (no month
fixed effects -- weather is left to explain seasonality itself), and the
anomaly model (with month fixed effects -- the rigorous test of whether
weather matters *beyond* time of year).

In [ ]:
import io
import requests
import matplotlib.pyplot as plt
import statsmodels.api as sm


def get_iowa_asos_stations():
    """Iowa ASOS station metadata (id, lat/lon, online status)."""
    url = "https://mesonet.agron.iastate.edu/geojson/network/IA_ASOS.geojson"
    data = requests.get(url, timeout=30).json()

    rows = []
    for feat in data["features"]:
        props = feat["properties"]
        rows.append({"station": props["sid"], "online": props["online"]})

    stations_df = pd.DataFrame(rows)
    return stations_df[stations_df["online"]].reset_index(drop=True)


def fetch_iem_daily_weather(stations, start_date, end_date, network="IA_ASOS"):
    """Fetch daily weather summaries (incl. NOAA climate-normal columns) from IEM."""
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    url = "https://mesonet.agron.iastate.edu/cgi-bin/request/daily.py"
    params = {
        "network": network,
        "stations": ",".join(sorted(set(stations))),
        "year1": start_date.year, "month1": start_date.month, "day1": start_date.day,
        "year2": end_date.year, "month2": end_date.month, "day2": end_date.day,
        "format": "csv",
    }

    resp = requests.get(url, params=params, timeout=180)
    resp.raise_for_status()

    weather_df = pd.read_csv(io.StringIO(resp.text))
    weather_df["day"] = pd.to_datetime(weather_df["day"])
    return weather_df


stations_df = get_iowa_asos_stations()

start_date = df["year_month"].min() + "-01"
# last full or partial month present in the panel -- use the actual max ordered_on
# from the source table so we don't ask IEM for dates that don't exist yet
end_date = con.execute("SELECT MAX(ordered_on) FROM 'liquor_2022_2026.parquet'").fetchone()[0]

station_weather = fetch_iem_daily_weather(stations_df["station"], start_date, end_date)
print(station_weather.shape, station_weather["day"].min(), station_weather["day"].max())
station_weather.head()

In [ ]:
# statewide daily average across every reporting station that day
statewide_daily = station_weather.groupby("day").agg(
    avg_max_temp_f=("max_temp_f", "mean"),
    avg_min_temp_f=("min_temp_f", "mean"),
    precip_in=("precip_in", "mean"),
    snow_in=("snow_in", "mean"),
    climo_high_f=("climo_high_f", "mean"),
    climo_precip_in=("climo_precip_in", "mean"),
).reset_index()

statewide_daily["temp_anomaly_f"] = statewide_daily["avg_max_temp_f"] - statewide_daily["climo_high_f"]
statewide_daily["precip_anomaly_in"] = statewide_daily["precip_in"] - statewide_daily["climo_precip_in"]

statewide_daily["year_month"] = statewide_daily["day"].dt.strftime("%Y-%m")

monthly_weather = statewide_daily.groupby("year_month").agg(
    avg_max_temp_f=("avg_max_temp_f", "mean"),
    avg_min_temp_f=("avg_min_temp_f", "mean"),
    total_precip_in=("precip_in", "sum"),
    total_snow_in=("snow_in", "sum"),
    avg_temp_anomaly_f=("temp_anomaly_f", "mean"),
    total_precip_anomaly_in=("precip_anomaly_in", "sum"),
).reset_index()

print(monthly_weather.shape)
monthly_weather.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(monthly_weather["year_month"], monthly_weather["avg_max_temp_f"], marker="o", markersize=3)
axes[0].set_ylabel("Avg Max Temp (F)")
axes[0].set_title("Statewide Monthly Weather, 2022-2026")

axes[1].bar(monthly_weather["year_month"], monthly_weather["avg_temp_anomaly_f"],
            color=np.where(monthly_weather["avg_temp_anomaly_f"] >= 0, "firebrick", "steelblue"))
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Temp Anomaly vs Normal (F)")

for ax in axes:
    ax.tick_params(axis="x", rotation=90)
    for label in ax.get_xticklabels()[::3]:
        label.set_visible(True)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % 3 != 0:
            label.set_visible(False)

plt.tight_layout()
plt.show()

### Merge weather onto the category x month panel, rebuild `model_df`

`model_df` from the cell above was built before weather existed — rebuilding
it here now that `monthly_weather` is available. Also dropping the blank
`category_name` row (uncategorized/miscoded sales that don't belong to any
real product category, and would otherwise get their own meaningless dummy
column).

In [ ]:
df_weather = df.merge(monthly_weather, on="year_month", how="left")

print("rows before weather merge:", len(df))
print("rows with no weather match:", df_weather["avg_max_temp_f"].isna().sum())

df_weather = df_weather[df_weather["category_name"].str.strip() != ""].copy()

# rebuild the linear time trend + dummy design matrix now that weather columns exist
df_weather["t"] = (df_weather["year"] - df_weather["year"].min()) * 12 + df_weather["month_num"]

model_df = pd.get_dummies(
    df_weather,
    columns=["month_num", "category_name"],
    drop_first=True,
)

print(model_df.shape)
model_df.head()

### Model A — raw weather levels, no month fixed effects

Weather is left to explain seasonality on its own here (no month dummies).
This is the more intuitive model to describe to DEAD ("warmer months look
like this"), but note it can't distinguish a pure calendar effect from an
actual weather effect.

In [ ]:
category_dummy_cols = [c for c in model_df.columns if c.startswith("category_name_")]

feature_cols_a = ["avg_max_temp_f", "total_precip_in", "t"] + category_dummy_cols
X_a = sm.add_constant(model_df[feature_cols_a].astype(float))
y = model_df["log_units"].astype(float)

model_a = sm.OLS(y, X_a, missing="drop").fit()
print(model_a.summary())

### Model B — weather *anomalies*, with month fixed effects (the rigorous test)

Now let month dummies soak up "what time of year is it," and ask whether a
month's temperature/precipitation being unusual *for that month* still
moves sales. This is the model that actually speaks to DEAD's hypothesis
("does weather drive purchases," as opposed to "does time of year").

In [ ]:
month_dummy_cols = [c for c in model_df.columns if c.startswith("month_num_")]

feature_cols_b = ["avg_temp_anomaly_f", "total_precip_anomaly_in", "t"] + month_dummy_cols + category_dummy_cols
X_b = sm.add_constant(model_df[feature_cols_b].astype(float))

model_b = sm.OLS(y, X_b, missing="drop").fit()
print(model_b.summary())

In [ ]:
comparison = pd.DataFrame({
    "Model A (raw weather, no month FE)": {
        "temp coef (per deg F)": model_a.params.get("avg_max_temp_f"),
        "temp p-value": model_a.pvalues.get("avg_max_temp_f"),
        "precip coef (per inch)": model_a.params.get("total_precip_in"),
        "precip p-value": model_a.pvalues.get("total_precip_in"),
        "R-squared": model_a.rsquared,
    },
    "Model B (temp/precip anomaly, +month FE)": {
        "temp coef (per deg F)": model_b.params.get("avg_temp_anomaly_f"),
        "temp p-value": model_b.pvalues.get("avg_temp_anomaly_f"),
        "precip coef (per inch)": model_b.params.get("total_precip_anomaly_in"),
        "precip p-value": model_b.pvalues.get("total_precip_anomaly_in"),
        "R-squared": model_b.rsquared,
    },
})
comparison

### Which categories move most with temperature?

A quick, model-free cut: correlate each category's month-to-month `log_units`
with that month's temperature anomaly, to see whether a warm-weather effect
(if any) concentrates in particular products (e.g. beer/RTD/tequila) rather
than liquor sales broadly.

In [ ]:
category_temp_corr = (
    df_weather
    .groupby("category_name")[["log_units", "avg_temp_anomaly_f"]]
    .apply(lambda g: g["log_units"].corr(g["avg_temp_anomaly_f"]) if len(g) > 5 else np.nan)
    .dropna()
    .sort_values(ascending=False)
)

print("Most positively correlated with warm-anomaly months:")
print(category_temp_corr.head(10))
print()
print("Most negatively correlated with warm-anomaly months:")
print(category_temp_corr.tail(10))

### Interpretation for DEAD

- **Model A** (raw weather, no seasonality control) will show a strong,
  "significant" temperature effect almost by construction — it's largely
  just re-detecting that summer sells more than winter. Useful for a plain
  "warmer months, more sales" chart, but not proof that weather itself is
  the driver.
- **Model B** (anomaly, with month fixed effects) is the harder test: it
  asks whether an *unusually* warm or wet month, relative to that month's
  own normal, still moves sales after controlling for which month it is.
  Look at `avg_temp_anomaly_f`'s coefficient, sign, and p-value in the
  summary above to see whether that survives.
- Caveat to give DEAD directly: this panel has 56 months and 52 categories,
  so month-to-month weather variation (temperature anomalies of a few
  degrees) is a much smaller signal than the enormous swing between, say,
  January and July, or the Nov/Dec holiday spike documented in
  `final_eda.ipynb`. A null or weak result on `avg_temp_anomaly_f` doesn't
  mean weather is irrelevant to drinking behavior generally — it means
  *month-to-month weather noise*, specifically, isn't a strong lever
  relative to calendar/holiday effects at this level of aggregation. If
  DEAD's real interest is short-term spikes (e.g. a heat wave weekend),
  that calls for the daily/zip-level data from `final_eda.ipynb` instead of
  this monthly panel.
- The category-level correlations above are a starting point for "does the
  weather effect concentrate somewhere" (e.g. patio/RTD drinks vs. spirits
  meant for cold-weather sipping) — worth a follow-up interaction model if
  DEAD wants to push on that specifically.